# GameTheory-02c-Travelers-Dilemma-Rust

> **Navigation** : [Notebooks](../) | [Série GameTheory](README.md) | [<< 2-NormalForm (palier parent)](GameTheory-02-NormalForm-Python.ipynb) | [2b - Définitions Lean](GameTheory-02b-Lean-Definitions-Lean.ipynb) | [Version Python](GameTheory-02c-Travelers-Dilemma-Python.ipynb) | [Version C#](GameTheory-02c-Travelers-Dilemma-CSharp.ipynb)

> **Série** : GameTheory-02c-Travelers-Dilemma-Rust — accrétion du palier 02 (forme normale, dominance, élimination itérée) : le Dilemme des Voyageurs de Basu (1994)
## Prérequis
- [02-NormalForm (palier parent)](GameTheory-02-NormalForm-Python.ipynb) — forme normale, dominance, élimination itérée


## Le paradigme : un accident de compagnie aérienne

Deux voyageurs reviennent d'une expédition. La compagnie a perdu les **deux bagages**, dont le contenu est **identique**. Le directeur de la compagnie leur demande de soumettre chacun, **indépendamment** et **sans communiquer**, une valeur de réclamation entre **2** et **100** dollars.

La règle d'indemnisation est astucieuse :

* Le directeur paie aux deux le **minimum** des deux réclamations.
* Celui qui a écrit le **plus petit** montant reçoit un **bonus** de **2 $** pour son honnêteté.
* Celui qui a écrit le **plus grand** montant subit une **pénalité** de **2 $** pour avoir exagéré.

> **Basu (1994), "The Traveler's Dilemma: Paradoxes of Rationality in Game Theory", *American Economic Review* 84(2), 391-395.**


### Pourquoi ce jeu est un paradigme

Ce jeu est l'un des objets les plus purs du **paradoxe de la rationalité** en théorie des jeux : il n'a ni répétition dans le temps, ni non-convexité cachée — seulement une forme normale à deux joueurs, stratégies bornées, gain continu par morceaux. Et pourtant, la raison formelle et l'intuition humaine divergent **radicalement** :

* La théorie prédit que **deux voyageurs rationnels réclament 2 $** (l'équilibre de Nash).
* Les joueurs humains, eux, réclament le plus souvent **100 $** (ou un montant proche).

Le but du notebook est de **montrer les deux côtés** : construire le raisonnement d'élimination qui mène à (2,2), puis regarder ce qui se passe quand on **varie le bonus $r$** — et voir que le paradoxe, lui, a un **seuil précis** en dessous duquel il disparaît.


### Source primaire

L'énoncé formel et la structure de paiement sont lus **firsthand** dans l'article de Basu (1994). La version standard (r=2, bornes 2..100) est celle reproduite ici. La forme générale de la fonction de paiement paramétrée par le bonus $r$ (et le plafond $\bar c$) est celle qui permet l'analyse de sensibilité du §4.

| Élément | Valeur |
|---|---|
| Bornes de réclamation | $c_i \in [2, 100]$ |
| Bonus d'honnêteté | $r = 2$ |
| Pénalité d'exagération | $r = 2$ |
| Lien | [JSTOR 2117865](https://www.jstor.org/stable/2117865) |


## 1. La fonction de paiement : l'objet formel


### La règle de l'arbitre, en une expression

Soient $x$ la réclamation du joueur Ligne et $y$ celle du joueur Colonne, $r$ le bonus/pénalité, $\bar c$ le plafond commun ($\bar c = 100$ ici). Le paiement de Ligne vaut :

$$\nu_1(x, y) = \begin{cases}
   \min(x,y) + r & \text{si } x < y \\[2pt]
   \min(x,y) - r & \text{si } x > y \\[2pt]
   x & \text{si } x = y
\end{cases}
$$

La fonction est symétrique pour le joueur 2. On la code directement, avec `r` en paramètre — c'est ce qui permettra la sensibilité du §4.


### Fonction de paiement


In [2]:
// Portee de la sensibilite : bonus r variable, plafond fixe
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Nombre a precision choisie : entier quand r est entier, flottant sinon
#[derive(Clone, Copy, Debug)]
enum Num {
    I(i64),
    F(f64),
}

impl Num {
    fn add(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a + b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 + b),
            (Num::F(a), Num::I(b)) => Num::F(a + b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a + b),
        }
    }
    fn sub(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a - b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 - b),
            (Num::F(a), Num::I(b)) => Num::F(a - b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a - b),
        }
    }
}

// Affichage : entier nu pour I, repr Debug pour F (101.0, 0.0, -0.5)
impl std::fmt::Display for Num {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        match self {
            Num::I(n) => write!(f, "{}", n),
            Num::F(v) => write!(f, "{:?}", v),
        }
    }
}

// Paiement du joueur Ligne. x et y dans [C_MIN, C_MAX].
fn u1(x: i64, y: i64, r: Num) -> Num {
    let m = x.min(y);
    if x < y {
        return Num::I(m).add(r);
    }
    if x > y {
        return Num::I(m).sub(r);
    }
    Num::I(m)
}

// Paiement du joueur Colonne (symetrie).
fn u2(x: i64, y: i64, r: Num) -> Num {
    u1(y, x, r)
}

println!("u1(90, 100, r=2) = {}    (90 < 100 -> min=90 + bonus 2)", u1(90, 100, Num::I(2)));
println!("u1(100, 90, r=2) = {}    (100 > 90 -> min=90 - penalite 2)", u1(100, 90, Num::I(2)));
println!("u1(90, 90, r=2)  = {}    (egalite -> 90)", u1(90, 90, Num::I(2)));


u1(90, 100, r=2) = 92    (90 < 100 -> min=90 + bonus 2)


u1(100, 90, r=2) = 88    (100 > 90 -> min=90 - penalite 2)


u1(90, 90, r=2)  = 90    (egalite -> 90)


### Lecture du résultat

Trois cas se lisent comme attendu : réclamer **moins** que l'autre rapporte le minimum **majoré** du bonus ; réclamer **plus** le minimum **minoré** de la pénalité ; l'égalité donne le montant commun. La dissymétrie est le moteur de tout le paradoxe : baisser sa réclamation est toujours **au moins** aussi bon que la maintenir, dès lors que l'autre est en dessous.


## 2. La meilleure réponse : se tenir juste en dessous de l'autre


### Définition

La **meilleure réponse** de Ligne à une réclamation $y$ de Colonne est la valeur de $x$ qui maximise $u_1(x, y)$. On l'énumère directement sur la grille bornée.


In [3]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Nombre a precision choisie : entier quand r est entier, flottant sinon
#[derive(Clone, Copy, Debug)]
enum Num {
    I(i64),
    F(f64),
}

impl Num {
    fn add(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a + b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 + b),
            (Num::F(a), Num::I(b)) => Num::F(a + b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a + b),
        }
    }
    fn sub(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a - b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 - b),
            (Num::F(a), Num::I(b)) => Num::F(a - b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a - b),
        }
    }
    // Comparaison au bit : passage par f64 (valeurs exactes ici)
    fn f(&self) -> f64 {
        match self {
            Num::I(n) => *n as f64,
            Num::F(v) => *v,
        }
    }
}

// Affichage : entier nu pour I, repr Debug pour F (101.0, 0.0, -0.5)
impl std::fmt::Display for Num {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        match self {
            Num::I(n) => write!(f, "{}", n),
            Num::F(v) => write!(f, "{:?}", v),
        }
    }
}

// Paiement du joueur Ligne. x et y dans [C_MIN, C_MAX].
fn u1(x: i64, y: i64, r: Num) -> Num {
    let m = x.min(y);
    if x < y {
        return Num::I(m).add(r);
    }
    if x > y {
        return Num::I(m).sub(r);
    }
    Num::I(m)
}

// Meilleure reponse de Ligne a la reclamation y de Colonne.
//
// En cas d'egalite de paiement, on parcourt x par x croissant et on garde
// le PREMIER maximum atteint (comparaison stricte) : la convention
// algorithmique retient le plus petit argmax. Ce n'est pas une unicite
// mathematique -- a r = 1 la vraie meilleure reponse est multi-valuee
// {y-1, y} (cf. section 4).
fn best_response(y: i64, r: Num) -> (i64, Num) {
    let mut best_x = C_MIN;
    let mut best_v = u1(C_MIN, y, r);
    for x in (C_MIN + 1)..=C_MAX {
        let v = u1(x, y, r);
        if v.f() > best_v.f() {
            best_x = x;
            best_v = v;
        }
    }
    (best_x, best_v)
}

for y in [100i64, 2, 55, 3] {
    let (x_star, v_star) = best_response(y, Num::F(2.0));
    println!("y={:<3} -> x* ({}, {})", y, x_star, v_star);
};

y=100 -> x* (99, 101.0)


y=2   -> x* (2, 2)


y=55  -> x* (54, 56.0)


y=3   -> x* (2, 4.0)


### Lecture du résultat

La meilleure réponse est **systématiquement `max(C_MIN, y-1)`** : réclamer *un de moins* que l'autre donne le minimum (=$y-1$) **plus** le bonus, soit $y-1+2 = y+1$, ce qui bat réclamer la même chose ($y$) ou davantage. Seule exception : quand l'autre a déjà réclamé le **plancher** ($y=2$), on ne peut pas descendre en dessous — on est forcé à l'égalité (2,2).

Ce calcul est fait à $r = 2$. À $r = 1$ il change de nature : sous-coter à $y-1$ rapporte $(y-1)+1 = y$, soit **exactement autant** que rester à $y$ — la meilleure réponse devient l'**ensemble** $\{y-1,\, y\}$, et la cellule ci-dessus n'en montre qu'un représentant (le plus petit, par convention de tie-breaking).


## 3. L'élimination itérée des stratégies faiblement dominées


### Le raisonnement qui mène à (2,2)

La meilleure réponse à une réclamation $y$ est **$y-1$** : réclamer un de moins que l'autre rapporte $y-1+r$ (le minimum majoré du bonus), ce qui bat réclamer la même chose ($y$) ou davantage. La conséquence n'est pas un gain global immédiat, mais une **élimination de haut en bas** : dans le jeu réduit aux stratégies restantes, la plus haute réclamation $k$ est **faiblement dominée** par $k-1$ — jamais *strictement* dominée, car face à un adversaire qui réclame $y < k-1$, les deux stratégies sont également pénalisées ($y - r$ chacune). La domination est *faible au sens exact* : $k-1$ rapporte **au moins autant partout** (égalité stricte contre $y < k-1$) et **strictement plus quelque part** (contre $y = k-1$, où $k$ écope de la pénalité ; contre $y = k$ dès que $r > 1$). En cascade, 100 est éliminé, puis 99, … jusqu'à ce qu'il ne reste que **$x = 2$**.


In [4]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Nombre a precision choisie : entier quand r est entier, flottant sinon
#[derive(Clone, Copy, Debug)]
enum Num {
    I(i64),
    F(f64),
}

impl Num {
    fn add(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a + b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 + b),
            (Num::F(a), Num::I(b)) => Num::F(a + b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a + b),
        }
    }
    fn sub(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a - b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 - b),
            (Num::F(a), Num::I(b)) => Num::F(a - b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a - b),
        }
    }
    fn f(&self) -> f64 {
        match self {
            Num::I(n) => *n as f64,
            Num::F(v) => *v,
        }
    }
}

// Affichage : entier nu pour I, repr Debug pour F
impl std::fmt::Display for Num {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        match self {
            Num::I(n) => write!(f, "{}", n),
            Num::F(v) => write!(f, "{:?}", v),
        }
    }
}

// Paiement du joueur Ligne. x et y dans [C_MIN, C_MAX].
fn u1(x: i64, y: i64, r: Num) -> Num {
    let m = x.min(y);
    if x < y {
        return Num::I(m).add(r);
    }
    if x > y {
        return Num::I(m).sub(r);
    }
    Num::I(m)
}

// Paiement du joueur Colonne (symetrie).
fn u2(x: i64, y: i64, r: Num) -> Num {
    u1(y, x, r)
}

// Elimination iteree des strategies faiblement dominees (de haut en bas).
//
// Dans le jeu reduit a l'ensemble `remaining`, la strategie x est
// FAIBLEMENT dominee par x-1 si x-1 rapporte au moins autant partout
// (sur `remaining`) ET strictement plus quelque part. Ce n'est jamais une
// domination stricte : contre les reclamations basses y < x-1, x-1 et x
// sont egalement penalisees (y - r chacune).
fn weakly_dominated_by_prev(x: i64, remaining: &[i64], r: Num) -> bool {
    if !remaining.contains(&(x - 1)) {
        return false;
    }
    let mut weakly = true;
    let mut strictly = false;
    for &y in remaining {
        let a = u1(x - 1, y, r).f();
        let b = u1(x, y, r).f();
        if a < b {
            weakly = false;
        }
        if a > b {
            strictly = true;
        }
    }
    weakly && strictly
}

fn iteratively_dominated(c_max: i64) -> (Vec<i64>, Vec<i64>) {
    let r = Num::F(2.0);
    let mut remaining: Vec<i64> = (C_MIN..=c_max).collect();
    let mut eliminated: Vec<i64> = Vec::new();
    loop {
        let candidates: Vec<i64> = remaining
            .iter()
            .copied()
            .filter(|&x| x > C_MIN && weakly_dominated_by_prev(x, &remaining, r))
            .collect();
        if candidates.is_empty() {
            break;
        }
        let to_eliminate = *candidates.iter().max().unwrap(); // on elimine la plus haute d'abord
        remaining.retain(|&v| v != to_eliminate);
        eliminated.push(to_eliminate);
    }
    (remaining, eliminated)
}

let (final_, order) = iteratively_dominated(C_MAX);
println!("Strategies eliminées (du haut vers le bas), extrait: {:?} ...", &order[..6]);
println!("Strategies survivantes: {:?}", final_);
println!(" -> pour r = 2 (canonique), (2,2) est l'unique equilibre de Nash ;");
println!("    a r = 1 l'unicite tombe (equilibres diagonaux multiples, cf. section 4).");
println!();
// Verification : a l'equilibre (2,2), aucun joueur ne veut bouger seul
println!("u1(2,2) = {}  | devier a 3 pour Ligne : {}  -> pas profitable", u1(2, 2, Num::F(2.0)), u1(3, 2, Num::F(2.0)));
println!("u1(2,2) = {}  | devier a 3 pour Colonne : {}  -> pas profitable", u1(2, 2, Num::F(2.0)), u2(2, 3, Num::F(2.0)));


Strategies eliminées (du haut vers le bas), extrait: [100, 99, 98, 97, 96, 95] ...


Strategies survivantes: [2]


 -> pour r = 2 (canonique), (2,2) est l'unique equilibre de Nash ;


    a r = 1 l'unicite tombe (equilibres diagonaux multiples, cf. section 4).


u1(2,2) = 2  | devier a 3 pour Ligne : 0.0  -> pas profitable


u1(2,2) = 2  | devier a 3 pour Colonne : 0.0  -> pas profitable


### Lecture du résultat

L'élimination itérée ne laisse qu'une seule stratégie : **réclamer 2 \$**. À (2,2), chaque joueur reçoit 2 \$, et s'écarter seul (monter) fait passer à $\min=2 - r$ : **0 \$**, donc strictement pire. Pour $r > 1$ — le cas canonique $r=2$ comme tout $r$ plus grand — l'équilibre de Nash unique du jeu est donc **$(2,2)$**, et c'est la prédiction formelle. À $r = 1$ exactement, cette unicité tombe : chaque profil diagonal $(y,y)$ devient lui aussi un équilibre de Nash (section 4 le mesure). C'est l'apogée du paradoxe : la théorie dit « le minimum », l'intuition humaine dit autre chose (voir §4 et §5).


## 4. La sensibilité au bonus : où naît le paradoxe


### Le paradoxe dépend de $r$

Le raisonnement d'élimination ci-dessus distingue **trois régimes** selon le bonus :

* **$r > 1$** : le sous-cotage est *strictement* profitable — comparer la plus haute réclamation $k$ à $k-1$ face à un adversaire en $k$ donne $k-1+r > k$. La spirale descend jusqu'au plancher et $(2,2)$ est l'**équilibre unique**.
* **$r = 1$** : sous-coter à $y-1$ et rester à $y$ rapportent **autant** ($y$ chacun). La meilleure réponse est multi-valuée $\{y-1, y\}$, chaque profil diagonal $(y,y)$ est un équilibre de Nash, et l'élimination — qui repose sur la domination *faible* — ne descend jusqu'au plancher que moyennant la convention de tie-breaking. L'unicité de $(2,2)$ n'est plus un théorème : c'est un artefact d'ordre.
* **$r < 1$** : le gain de sous-cotage est négatif — l'élimination de haut en bas **ne démarre même pas**.

L'expérience — humaine et expérimentale — montre que les gens jouent haut **quand $r$ est petit** et bas **quand $r$ est grand**. Le paradoxe n'est pas un fait : c'est une fonction de $r$.

On mesure la différence entre l'équilibre théorique (2) et la prédiction « naïve » qu'un joueur stratégiquement **superficiel** ferait : viser le maximum $\bar c$, puisqu'il s'agit avant tout de capter le bonus.


In [5]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Nombre a precision choisie : entier quand r est entier, flottant sinon
#[derive(Clone, Copy, Debug)]
enum Num {
    I(i64),
    F(f64),
}

impl Num {
    fn add(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a + b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 + b),
            (Num::F(a), Num::I(b)) => Num::F(a + b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a + b),
        }
    }
    fn sub(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a - b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 - b),
            (Num::F(a), Num::I(b)) => Num::F(a - b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a - b),
        }
    }
    fn f(&self) -> f64 {
        match self {
            Num::I(n) => *n as f64,
            Num::F(v) => *v,
        }
    }
}

// Affichage : entier nu pour I, repr Debug pour F (0.5, -0.5, 10.0)
impl std::fmt::Display for Num {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        match self {
            Num::I(n) => write!(f, "{}", n),
            Num::F(v) => write!(f, "{:?}", v),
        }
    }
}

// Paiement du joueur Ligne. x et y dans [C_MIN, C_MAX].
fn u1(x: i64, y: i64, r: Num) -> Num {
    let m = x.min(y);
    if x < y {
        return Num::I(m).add(r);
    }
    if x > y {
        return Num::I(m).sub(r);
    }
    Num::I(m)
}

// Gain a devier de c vers c-1 quand l'autre tient c : u1(c-1,c) - u1(c,c).
// Independante de c : vaut (c-1+r) - c = r - 1. C'est le moteur de la spirale.
fn undercut_gain(c: i64, r: Num) -> Num {
    u1(c - 1, c, r).sub(u1(c, c, r))
}

// Domination FAIBLE de x par x-1 dans le jeu reduit a `remaining`.
fn weakly_dominated_by_prev(x: i64, remaining: &[i64], r: Num) -> bool {
    if !remaining.contains(&(x - 1)) {
        return false;
    }
    let mut weakly = true;
    let mut strictly = false;
    for &y in remaining {
        let a = u1(x - 1, y, r).f();
        let b = u1(x, y, r).f();
        if a < b {
            weakly = false;
        }
        if a > b {
            strictly = true;
        }
    }
    weakly && strictly
}

// L'elimination de haut en bas atteint-elle le plancher pour ce bonus r ?
// Reutilise le raisonnement de la cellule 3 (domination FAIBLE de la plus
// haute restante par son precedesseur), en propageant r partout.
fn elimination_reaches_floor(r: Num) -> bool {
    let mut remaining: Vec<i64> = (C_MIN..=C_MAX).collect();
    loop {
        let candidates: Vec<i64> = remaining
            .iter()
            .copied()
            .filter(|&x| x > C_MIN && weakly_dominated_by_prev(x, &remaining, r))
            .collect();
        if candidates.is_empty() {
            break;
        }
        let to_eliminate = *candidates.iter().max().unwrap();
        remaining.retain(|&v| v != to_eliminate);
    }
    remaining == vec![C_MIN]
}

// TOUTES les meilleures reponses a y (pas seulement le plus petit argmax).
fn best_response_set(y: i64, r: Num) -> Vec<i64> {
    let mut top = u1(C_MIN, y, r).f();
    for x in (C_MIN + 1)..=C_MAX {
        let v = u1(x, y, r).f();
        if v > top {
            top = v;
        }
    }
    let mut out: Vec<i64> = Vec::new();
    for x in C_MIN..=C_MAX {
        if u1(x, y, r).f() == top {
            out.push(x);
        }
    }
    out
}

// (y, y) est-il un equilibre de Nash ? Aucun des deux ne gagne a devier.
fn is_nash_diagonal(y: i64, r: Num) -> bool {
    let me = u1(y, y, r).f();
    (C_MIN..=C_MAX).all(|x| u1(x, y, r).f() <= me)
}

// Affichage d'une liste d'entiers : [2] ou [99, 100]
fn fmt_list(xs: &[i64]) -> String {
    let parts: Vec<String> = xs.iter().map(|x| x.to_string()).collect();
    format!("[{}]", parts.join(", "))
}

// Le moteur du paradoxe : le gain de sous-cotage r-1, et le seuil r*=1.
let rs = [0.5f64, 1.0, 1.5, 2.0, 5.0, 10.0];
println!("  r  |  gain de sous-cotage (r-1)  |  la spirale atteint (2,2) ?");
for &r in rs.iter() {
    let g = undercut_gain(50, Num::F(r));
    let ok = elimination_reaches_floor(Num::F(r));
    println!("{:>4} | {:>8} | {}", format!("{:?}", r), g.to_string(), if ok { "OUI" } else { "NON" });
}

println!();
// Cas a part r = 1 : meilleure reponse multi-valuee + equilibres diagonaux
for &y in [100i64, 55, 3, 2].iter() {
    println!("r=1 :: BR({}) = {}", y, fmt_list(&best_response_set(y, Num::F(1.0))));
}
let diag_r1: Vec<i64> = (C_MIN..=C_MAX).filter(|&y| is_nash_diagonal(y, Num::F(1.0))).collect();
let diag_r2: Vec<i64> = (C_MIN..=C_MAX).filter(|&y| is_nash_diagonal(y, Num::F(2.0))).collect();
println!("r=1 :: equilibres de Nash diagonaux : {} au total, de {} a {}", diag_r1.len(), diag_r1[0], diag_r1[diag_r1.len() - 1]);
println!("r=2 :: equilibres de Nash diagonaux : {}", fmt_list(&diag_r2));

println!();
println!("Lecture : le gain de sous-cotage vaut r-1, independant du point de depart c.");
println!("  -> r < 1 : sous-coter ne rapporte rien (gain negatif) -> grimper n'est pas puni,");
println!("             la spirale ne demarre pas -> le paradoxe est VIVANT (humains jouent haut).");
println!("  -> r = 1 : egalites partout -- BR multi-valuee {{y-1, y}}, chaque (y,y) est un");
println!("             equilibre de Nash ; l'elimination faible n'atteint le plancher que");
println!("             par convention d'ordre. PAS d'equilibre unique.");
println!("  -> r > 1 : sous-coter devient strictement rentable -> la spirale descend");
println!("             forcement vers (2,2), unique equilibre -> le paradoxe SE DISSOUT.");

  r  |  gain de sous-cotage (r-1)  |  la spirale atteint (2,2) ?


 0.5 |     -0.5 | NON


 1.0 |      0.0 | OUI


 1.5 |      0.5 | OUI


 2.0 |      1.0 | OUI


 5.0 |      4.0 | OUI


... (11 outputs left out)

             equilibre de Nash ; l'elimination faible n'atteint le plancher que


             par convention d'ordre. PAS d'equilibre unique.


  -> r > 1 : sous-coter devient strictement rentable -> la spirale descend


             forcement vers (2,2), unique equilibre -> le paradoxe SE DISSOUT.


### Lecture du résultat

Le moteur du paradoxe est le **gain de sous-cotage** : si l'autre réclame $c$, réclamer $c-1$ rapporte $(c-1)+r$ au lieu de $c$ (l'égalité), soit un gain de $r-1$ — **indépendant** du point de départ. Ce gain unique pilote toute la spirale.

* **$r < 1$** : le gain de sous-cotage est négatif — sous-coter ne rapporte rien, grimper n'est pas puni. L'élimination de haut en bas **ne démarre même pas** : un joueur peut rationnellement rester haut, et les humains jouent en effet ~100 $. **Le paradoxe est vivant.**
* **$r = 1$** : le gain est exactement nul — sous-coter et rester rapportent autant. La meilleure réponse est multi-valuée $\{y-1, y\}$, et la sonde ci-dessus exhibe **chaque profil diagonal $(y,y)$ comme équilibre de Nash** ($99$ d'entre eux, contre un seul à $r=2$). L'élimination faible élimine bien tout jusqu'au plancher, mais elle écarte ainsi des stratégies qui sont d'authentiques meilleures réponses : l'unicité de $(2,2)$ est une **convention d'ordre**, pas un théorème.
* **$r > 1$** : le gain est positif — sous-coter devient strictement rentable à chaque étage, la spirale descend **forcément** jusqu'à (2,2), et la prédiction théorique coïncide avec le comportement. **Le paradoxe se dissout.**

Le seuil est donc **$r^* = 1$** (dans la version canonique $r=2$, on est bien au-dessus). C'est exactement le fait expérimental de Basu : la gravité du paradoxe est une fonction de $r$, pas une constante.


### Le moteur du paradoxe, en forme exacte

La section 4 affiche le **gain de sous-cotage** en flottant et l'affirme en prose : « le gain vaut $r-1$, indépendant du point de départ ». Ce bloc l'exhibe en **fraction exacte** — la forme précise que le flottant masque. Pour la version canonique $r=2$, le gain est **exactement** l'entier $1$, pas l'approximation $1.0$.


In [6]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Temoign EXACT du moteur du paradoxe : le gain de sous-cotage, en Frac.
// Copie pedagogique declaree : meme regle que u1 (cellule 1), en exact.
#[derive(Clone, Copy, Debug)]
struct Frac {
    n: i64,
    d: i64,
}

fn gcd(a: i64, b: i64) -> i64 {
    if b == 0 {
        a.abs()
    } else {
        gcd(b, a % b)
    }
}

impl Frac {
    // Le denominateur nul est invalide et doit etre signale explicitement.
    fn new(n: i64, d: i64) -> Result<Frac, String> {
        if d == 0 {
            return Err("denominateur nul".to_string());
        }
        let (n, d) = if d < 0 { (-n, -d) } else { (n, d) };
        let g = gcd(n, d);
        let g = if g == 0 { 1 } else { g };
        Ok(Frac { n: n / g, d: d / g })
    }

    fn int(m: i64) -> Frac {
        Frac { n: m, d: 1 }
    }

    fn add(self, o: Frac) -> Result<Frac, String> {
        Frac::new(self.n * o.d + o.n * self.d, self.d * o.d)
    }

    fn sub(self, o: Frac) -> Result<Frac, String> {
        Frac::new(self.n * o.d - o.n * self.d, self.d * o.d)
    }
}

// Affichage : n/d, ou n seul si le denominateur vaut 1
impl std::fmt::Display for Frac {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        if self.d == 1 {
            write!(f, "{}", self.n)
        } else {
            write!(f, "{}/{}", self.n, self.d)
        }
    }
}

fn u1_exact(x: i64, y: i64, r: Frac) -> Result<Frac, String> {
    let m = x.min(y);
    if x < y {
        return Frac::int(m).add(r);
    }
    if x > y {
        return Frac::int(m).sub(r);
    }
    Ok(Frac::int(m))
}

// Gain a devier de c vers c-1 : u1_exact(c-1,c) - u1_exact(c,c). EXACT, = r - 1.
fn undercut_gain_exact(c: i64, r: Frac) -> Result<Frac, String> {
    let gain = u1_exact(c - 1, c, r)?.sub(u1_exact(c, c, r)?)?;
    Ok(gain)
}

// Version canonique r = 2 : le gain est EXACTEMENT l'entier 1, independant de c
// (a la main : 101 - 100 = 56 - 55 = 4 - 3 = 1).
let r = Frac::int(2);
println!("Gain de sous-cotage EXACT, r = 2 (version canonique) :");
for c in [100i64, 55, 3] {
    match (
        u1_exact(c - 1, c, r),
        u1_exact(c, c, r),
        undercut_gain_exact(c, r),
    ) {
        (Ok(a), Ok(b), Ok(gain)) => println!(
            "  c={:>3} : u1({},{}) - u1({},{}) = {} - {} = {}",
            c, c - 1, c, c, c, a, b, gain
        ),
        (Err(e), _, _) | (_, Err(e), _) | (_, _, Err(e)) => {
            println!("  c={:>3} : erreur de fraction : {}", c, e)
        }
    }
}
println!();
println!("Gain de sous-cotage EXACT = r - 1, pour chaque r de la table section 4 :");
for rq_result in [
    Frac::new(1, 2),
    Ok(Frac::int(1)),
    Frac::new(3, 2),
    Ok(Frac::int(2)),
    Ok(Frac::int(5)),
    Ok(Frac::int(10)),
] {
    match rq_result {
        Ok(rq) => match undercut_gain_exact(50, rq) {
            Ok(gain) => println!("  r = {} : gain = {}", rq, gain),
            Err(e) => println!("  r = {} : erreur de fraction : {}", rq, e),
        },
        Err(e) => println!("  r = erreur de fraction : {}", e),
    }
};

Gain de sous-cotage EXACT, r = 2 (version canonique) :


  c=100 : u1(99,100) - u1(100,100) = 101 - 100 = 1


  c= 55 : u1(54,55) - u1(55,55) = 56 - 55 = 1


  c=  3 : u1(2,3) - u1(3,3) = 4 - 3 = 1


Gain de sous-cotage EXACT = r - 1, pour chaque r de la table section 4 :


  r = 1/2 : gain = -1/2


  r = 1 : gain = 0


  r = 3/2 : gain = 1/2


  r = 2 : gain = 1


  r = 5 : gain = 4


  r = 10 : gain = 9


### Lecture du témoin exact

Le moteur de la spirale est un **entier** : à $r=2$, sous-coter d'un cran rapporte **exactement** $+1$, quel que soit le point de départ $c$ ($101-100 = 56-55 = 4-3 = 1$). La loi « gain $= r-1$ » est **exacte**, pas approchée : pour chaque $r$ de la table §4, la fraction exacte est $r-1$ ($-\tfrac12,\ 0,\ \tfrac12,\ 1,\ 4,\ 9$). Le flottant ne masquait ici qu'une écriture : la valeur du moteur est un entier net.


## 5. Le contre-claim : l'équilibre de l'EEI est réel, mais il n'est pas la prédiction comportementale


### La question que l'article ne tait pas

L'énoncé exact — « l'équilibre de Nash est (2,2) » — est **vrai**, mais il n'implique **pas** que des joueurs humains le choisissent. Le contre-claim ici n'est pas de contredire le théorème, mais de **délimiter sa portée** : ce que l'élimination itérée *prouve* (un équilibre unique) est distinct de ce qu'elle *prédit* (le comportement). L'écart est exhibé quand on relâche l'hypothèse de rationalité parfaite et de connaissance commune.

On le montre en simulant un **jeu où un joueur n'est pas certain** que l'autre descend au plancher : s'il existe une probabilité $\varepsilon$ que l'autre « grimpe », le meilleur-pari pour le joueur prudent devient de **grimper** — et l'équilibre (2,2) cesse d'être un point stable de la dynamique d'adaptation.


In [7]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Nombre a precision choisie : entier quand r est entier, flottant sinon
#[derive(Clone, Copy, Debug)]
enum Num {
    I(i64),
    F(f64),
}

impl Num {
    fn add(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a + b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 + b),
            (Num::F(a), Num::I(b)) => Num::F(a + b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a + b),
        }
    }
    fn sub(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a - b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 - b),
            (Num::F(a), Num::I(b)) => Num::F(a - b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a - b),
        }
    }
    fn f(&self) -> f64 {
        match self {
            Num::I(n) => *n as f64,
            Num::F(v) => *v,
        }
    }
}

// Affichage : entier nu pour I, repr Debug pour F
impl std::fmt::Display for Num {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        match self {
            Num::I(n) => write!(f, "{}", n),
            Num::F(v) => write!(f, "{:?}", v),
        }
    }
}

// Paiement du joueur Ligne. x et y dans [C_MIN, C_MAX].
fn u1(x: i64, y: i64, r: Num) -> Num {
    let m = x.min(y);
    if x < y {
        return Num::I(m).add(r);
    }
    if x > y {
        return Num::I(m).sub(r);
    }
    Num::I(m)
}

// Somme compensee (algorithme de Neumaier) : c'est ce que fait sum() de
// CPython >= 3.12 sur des flottants, pas une somme gauche-a-droite.
// t = s + x ; c += (s - t) + x si |s| >= |x|, sinon c += (x - t) + s ; s = t
// puis s + c en fin de boucle si c est non nul et fini.
fn py_sum(xs: &[f64]) -> f64 {
    let mut s: f64 = 0.0;
    let mut c: f64 = 0.0;
    for &x in xs {
        let t = s + x;
        if s.abs() >= x.abs() {
            c += (s - t) + x;
        } else {
            c += (x - t) + s;
        }
        s = t;
    }
    if c != 0.0 && c.is_finite() {
        s += c;
    }
    s
}

// Esperance de paiement si mon adversaire tire y ~ opponent_dist.
fn expected_u1_given_mix(my_x: i64, opponent_dist: &[(i64, f64)], r: Num) -> f64 {
    let terms: Vec<f64> = opponent_dist
        .iter()
        .map(|&(y, p)| p * u1(my_x, y, r).f())
        .collect();
    py_sum(&terms)
}

// Meilleure reponse integrale face a une distribution de l'autre.
// On garde le PREMIER maximum (comparaison stricte) sur x croissant.
fn aggregate_be(opponent_dist: &[(i64, f64)], r: Num) -> i64 {
    let mut best_x = C_MIN;
    let mut best_v = expected_u1_given_mix(C_MIN, opponent_dist, r);
    for x in (C_MIN + 1)..=C_MAX {
        let v = expected_u1_given_mix(x, opponent_dist, r);
        if v > best_v {
            best_x = x;
            best_v = v;
        }
    }
    best_x
}

// Hypotheses : une fraction eps de l'autre 'grimpe' (uniforme sur [c_max//2, c_max]),
// le reste joue l'equilibre (2).
for eps in [0.0f64, 0.05, 0.2, 0.5] {
    let mut dist: Vec<(i64, f64)> = Vec::new();
    dist.push((C_MIN, 1.0 - eps));
    for y in (C_MAX / 2)..=C_MAX {
        dist.push((y, eps / ((C_MAX / 2 + 1) as f64)));
    }
    let x_be = aggregate_be(&dist, Num::F(2.0));
    println!("eps={:>4.2} :: meilleure reponse de Ligne = {:>3}", eps, x_be);
}

println!();
println!("Lecture : des qu'une fraction non nulle de l'autre peut grimper, la meilleure");
println!("reponse saute du plancher (2) vers le haut. L'equilibre (2,2) exige la confiance");
println!("ABSOLUE dans la rationalite de l'autre — c'est sa condition de stabilite.");


eps=0.00 :: meilleure reponse de Ligne =   2


eps=0.05 :: meilleure reponse de Ligne =  96


eps=0.20 :: meilleure reponse de Ligne =  96


eps=0.50 :: meilleure reponse de Ligne =  96


Lecture : des qu'une fraction non nulle de l'autre peut grimper, la meilleure


reponse saute du plancher (2) vers le haut. L'equilibre (2,2) exige la confiance


ABSOLUE dans la rationalite de l'autre — c'est sa condition de stabilite.


### Verdict du contre-claim

Le contre-claim n'est pas « le théorème est faux » — il est **vrai**. Le contre-claim est : **l'équilibre de l'élimination itérée n'est pas une prédiction comportementale**. Il ne tient que sous connaissance commune de la rationalité. Dès qu'une probabilité $\varepsilon$ de « grimper » dépasse un seuil positif $\varepsilon^* \approx 0{,}03$, la meilleure réponse du joueur prudent saute vers le haut (la simulation ci-dessus l'exhibe) — donc (2,2) est un équilibre *fragile*, pas un attracteur. C'est exactement la leçon que le notebook de distillation devait faire émerger, et qui nourrit l'expérience ICT candidate (voir §6).

## 6. Exercices (3) — à compléter par l'étudiant

Les trois exercices ci-dessous sont des stubs **C.1** (ils s'exécutent sans erreur, mais laissent le raisonnement à l'étudiant). Contenu réel à produire, jamais une valeur fabriquée.


In [8]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Nombre a precision choisie : entier quand r est entier, flottant sinon
#[derive(Clone, Copy, Debug)]
enum Num {
    I(i64),
    F(f64),
}

impl Num {
    fn add(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a + b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 + b),
            (Num::F(a), Num::I(b)) => Num::F(a + b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a + b),
        }
    }
    fn sub(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a - b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 - b),
            (Num::F(a), Num::I(b)) => Num::F(a - b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a - b),
        }
    }
    fn f(&self) -> f64 {
        match self {
            Num::I(n) => *n as f64,
            Num::F(v) => *v,
        }
    }
}

// Affichage : entier nu pour I, repr Debug pour F
impl std::fmt::Display for Num {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        match self {
            Num::I(n) => write!(f, "{}", n),
            Num::F(v) => write!(f, "{:?}", v),
        }
    }
}

// Paiement du joueur Ligne. x et y dans [C_MIN, C_MAX].
fn u1(x: i64, y: i64, r: Num) -> Num {
    let m = x.min(y);
    if x < y {
        return Num::I(m).add(r);
    }
    if x > y {
        return Num::I(m).sub(r);
    }
    Num::I(m)
}

// Affichage d'un Option<bool> : None / True / False (jamais {:?}).
fn opt_bool(v: Option<bool>) -> String {
    match v {
        None => "None".to_string(),
        Some(b) => if b { "True".to_string() } else { "False".to_string() },
    }
}

// Exercice 1 : Transformer le jeu en jeu de coordination
// Objectif : montrer que le Dilemme des Voyageurs n'est PAS un jeu antagoniste
// (a somme nulle) : les paiements de (x, y) et de (y, x) sont en miroir, et un
// profil comme (50, 50) fait gagner les DEUX joueurs a la fois -- c'est ce qui
// fait le dilemme.
// Question : pour quelles valeurs de r le profil (x=50, y=50) est-il meilleur
// que (x=2, y=2) pour les DEUX joueurs a la fois ? Repondez en code.
//
// NB pedagogique : ce qui suit est un VRAI stub C.1 -- la fonction renvoie None
// tant que l'etudiant n'a pas ecrit le corps. Lire la sortie : le verdict sera
// None jusqu'a implementation.

// Retourne True s'il existe x commun > C_MIN tel que u1(x,x,r) > u1(2,2,r).
fn coordination_possible(_r: f64) -> Option<bool> {
    // --- TODO etudiant : implementer la logique ci-dessous ---
    // Indice : comparer u1(C_MIN, C_MIN, r) (paiement au plancher) a u1(x, x, r)
    // pour x dans [C_MIN+1, c_max]. Si UN x rapporte strictement plus aux DEUX
    // joueurs que le plancher, retourner True ; sinon False.
    None // a remplacer par l'implementation de l'etudiant
}

// Sortie neutre tant que le stub n'est pas implemente
let result_1 = coordination_possible(2.0);
println!("u1(2,2)    = {}   ([2,2] = paiement du plancher)", u1(2, 2, Num::I(2)));
println!("u1(100,100) = {}   ([100,100] = egalite au plafond)", u1(100, 100, Num::I(2)));
println!("coordination_possible() -> {}   (None tant que l'etudiant n'a pas implemente)", opt_bool(result_1));


u1(2,2)    = 2   ([2,2] = paiement du plancher)


u1(100,100) = 100   ([100,100] = egalite au plafond)


coordination_possible() -> None   (None tant que l'etudiant n'a pas implemente)


### Exercice 2 — le seuil de dissolution du paradoxe

Cherchez la valeur de $r$ au-delà de laquelle « jouer le plafond » (l'égalité au maximum) rapporte **strictement plus** que « jouer le plancher » — comparez $u_1(\bar c,\bar c)$ et $u_1(2,2)$, puis dites si le verdict dépend de $r$. Justifiez à partir de la définition de $u_1$ sur la diagonale $x = y$.


In [9]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Nombre a precision choisie : entier quand r est entier, flottant sinon
#[derive(Clone, Copy, Debug)]
enum Num {
    I(i64),
    F(f64),
}

impl Num {
    fn add(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a + b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 + b),
            (Num::F(a), Num::I(b)) => Num::F(a + b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a + b),
        }
    }
    fn sub(self, o: Num) -> Num {
        match (self, o) {
            (Num::I(a), Num::I(b)) => Num::I(a - b),
            (Num::I(a), Num::F(b)) => Num::F(a as f64 - b),
            (Num::F(a), Num::I(b)) => Num::F(a - b as f64),
            (Num::F(a), Num::F(b)) => Num::F(a - b),
        }
    }
    fn f(&self) -> f64 {
        match self {
            Num::I(n) => *n as f64,
            Num::F(v) => *v,
        }
    }
}

// Affichage : entier nu pour I, repr Debug pour F (101.0, 0.0, -0.5)
impl std::fmt::Display for Num {
    fn fmt(&self, f: &mut std::fmt::Formatter) -> std::fmt::Result {
        match self {
            Num::I(n) => write!(f, "{}", n),
            Num::F(v) => write!(f, "{:?}", v),
        }
    }
}

// Paiement du joueur Ligne. x et y dans [C_MIN, C_MAX].
fn u1(x: i64, y: i64, r: Num) -> Num {
    let m = x.min(y);
    if x < y {
        return Num::I(m).add(r);
    }
    if x > y {
        return Num::I(m).sub(r);
    }
    Num::I(m)
}

// Affichage d'un Option<bool> : None / True / False (jamais {:?}).
fn opt_bool(v: Option<bool>) -> String {
    match v {
        None => "None".to_string(),
        Some(b) => if b { "True".to_string() } else { "False".to_string() },
    }
}

// Exercice 2 : le seuil de dissolution du paradoxe
// Question : est-ce que l'egalite au plafond (100,100) bat toujours l'egalite au
// plancher (2,2), quel que soit r ? Repondez en code (la fonction est un VRAI
// stub C.1 -- elle renvoie None tant que l'etudiant ne l'a pas implementee).

// L'egalite au plafond (c_max) rapporte-t-elle plus que l'egalite au plancher (2) ?
fn plafond_bat_plancher(_r: f64) -> Option<bool> {
    // --- TODO etudiant : implementer ---
    // Indice : comparer u1(c_max, c_max, r) et u1(C_MIN, C_MIN, r). Si la premiere
    // est strictement plus grande, retourner True ; sinon False.
    None // a remplacer par l'implementation de l'etudiant
}

// Sortie neutre tant que le stub n'est pas implemente
for r in [0.5f64, 2.0, 10.0, 50.0] {
    println!("r={:>5.1} | plafond(100,100)={:>6} | plancher(2,2)={:>4} | plafond bat ? {}", r,
             u1(100, 100, Num::F(r)).to_string(), u1(2, 2, Num::F(r)).to_string(),
             opt_bool(plafond_bat_plancher(r)));
};

r=  0.5 | plafond(100,100)=   100 | plancher(2,2)=   2 | plafond bat ? None


r=  2.0 | plafond(100,100)=   100 | plancher(2,2)=   2 | plafond bat ? None


r= 10.0 | plafond(100,100)=   100 | plancher(2,2)=   2 | plafond bat ? None


r= 50.0 | plafond(100,100)=   100 | plancher(2,2)=   2 | plafond bat ? None


### Exercice 3 — le coût de l'optimisme

Reprenez le calcul de l'espérance du §5 et trouvez la **valeur de $\varepsilon$** à partir de laquelle la meilleure réponse d'un joueur prudent passe du plancher (2) à un montant **strictement supérieur à 2**. C'est le « seuil de bascule » de la confiance.


In [10]:
const C_MIN: i64 = 2;
const C_MAX: i64 = 100;

// Exercice 3 : seuil de bascule de la confiance
// Objectif : determiner la valeur de epsilon pour laquelle la meilleure reponse
// d'un joueur prudent SAUTE au-dessus du plancher. La fonction est un VRAI stub
// C.1 -- elle renvoie None tant que l'etudiant ne l'a pas implementee.

// Retourne la plus petite valeur de eps pour laquelle la meilleure reponse
// stricte depasse le plancher C_MIN.
fn seuil_bascule() -> Option<f64> {
    // --- TODO etudiant : implementer ---
    // Indice : construire la distribution mixte sur l'adversaire ((C_MIN, 1-eps) +
    // uniforme sur [c_max//2, c_max]) pour plusieurs valeurs de eps ; reprendre
    // dans cette cellule le calcul de la section 5, notamment l'appel
    // aggregate_be(&dist, r), puis retourner le plus petit eps pour lequel le
    // resultat depasse strictement C_MIN.
    None // a remplacer par l'implementation de l'etudiant
}

// Affichage d'un Option<f64> : None ou la valeur (jamais {:?} pour Some).
fn opt_f64(v: Option<f64>) -> String {
    match v {
        None => "None".to_string(),
        Some(x) => format!("{:?}", x),
    }
}

let s = seuil_bascule();
println!("Seuil de bascule eps* = {}   (None tant que l'etudiant n'a pas implemente)", opt_f64(s));
println!("Indice pour l'etudiant : voir fonction aggregate_BE au paragraphe 5 ; le seuil attendu est ~0.03.");

Seuil de bascule eps* = None   (None tant que l'etudiant n'a pas implemente)


Indice pour l'etudiant : voir fonction aggregate_BE au paragraphe 5 ; le seuil attendu est ~0.03.


## À retenir

* **L'objet formel** : un jeu à deux joueurs à stratégies bornées $[2,\bar c]$, paiement $\min(x,y)$ plus bonus $r$ au plus petit / pénalité $r$ au plus grand.
* **Le claim exact** : l'élimination itérée des stratégies **faiblement** dominées descend jusqu'au plancher dès que $r \ge 1$ ; l'équilibre $(2,2)$ qu'elle laisse est **unique pour $r > 1$** — à $r = 1$, chaque profil diagonal $(y,y)$ est aussi un équilibre de Nash, et la survie du seul plancher est un artefact de la convention de tie-breaking.
* **Le contre-claim** : cet équilibre n'est pas une prédiction comportementale — il ne tient que sous connaissance commune de la rationalité. Une probabilité $\varepsilon > \varepsilon^*$ de jouer haut fait **sauter** la meilleure réponse du joueur prudent vers le plafond. Le seuil $\varepsilon^* \approx 0.03$ (réponse attendue de l'exercice 3, encadrée par le balayage du § 5 où la meilleure réponse saute entre $\varepsilon = 0$ et $\varepsilon = 0.05$) borne la **fragilité** de (2,2) : au-delà d'une fraction même infime d'adversaire « grimpeur », (2,2) n'est plus la meilleure réponse.
* **Le seuil** : le paradoxe se dissout quand $r$ devient grand ($r > 1$, équilibre unique au plancher) ; à $r = 1$ exactement, la multiplicité des équilibres rend la prédiction indéterminée ; il n'est paradoxal que pour $r$ petit ($r < 1$, élimination ne démarre pas).

> **Expérience ICT candidate** : mesurer en TP le taux de « montée » (réclamation > 50) pour $r=2$ vs $r=25$, et confronter à la courbe bascule de l'exercice 3 — c'est un test falsifiable du contre-claim.

> **Palier parent** : [GameTheory-02-NormalForm-Python](GameTheory-02-NormalForm-Python.ipynb) (forme normale, dominance, élimination itérée) · **Sibling** : [GameTheory-02b-Lean-Definitions-Lean](GameTheory-02b-Lean-Definitions-Lean.ipynb) · **Série** : [GameTheory](README.md)


## Vérifiez votre compréhension

<details>
<summary>Q1 — À quelle condition l'élimination itérée des stratégies faiblement dominées descend-elle jusqu'au plancher ?</summary>

**Réponse** : dès que le bonus $r \ge 1$ — le gain de sous-cotage vaut $r-1$. Pour $r=2$ (cas canonique), seules survivent les stratégies du plancher (`Strategies survivantes: [2]`) et `(2,2)` est l'unique équilibre de Nash.
</details>

<details>
<summary>Q2 — Que se passe-t-il exactement à $r = 1$ ?</summary>

**Réponse** : l'unicité tombe. Le balayage de la section 4 renvoie `r=1 :: equilibres de Nash diagonaux : 99 au total, de 2 a 100` : chaque profil diagonal $(y,y)$ est alors aussi un équilibre, et la survie du seul plancher est un artefact de la convention de tie-breaking.
</details>

<details>
<summary>Q3 — Quel est le seuil de confiance au-delà duquel l'équilibre (2,2) n'est plus la meilleure réponse ?</summary>

**Réponse** : $\varepsilon^* \approx 0.03$. La cellule de la section 5 montre la bascule : `eps=0.00 :: meilleure reponse de Ligne =   2` puis `eps=0.05 :: meilleure reponse de Ligne =  96` — une fraction infime d'adversaire « grimpeur » suffit à faire sauter la meilleure réponse du plancher vers le haut.
</details>

## Pour aller plus loin

- [GameTheory-04-NashEquilibrium-Python](GameTheory-04-NashEquilibrium-Python.ipynb) — équilibres purs et mixtes au-delà du paradigme du plancher.
- [GameTheory-02-NormalForm-Python](GameTheory-02-NormalForm-Python.ipynb) — forme normale, dominance et élimination itérée en général.
- Index de la série : [GameTheory](README.md).